<a href="https://colab.research.google.com/github/ADwiS777/Data-Cleansing-Layanan-ART/blob/main/2418040DataCleansing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import Libraries Dan Baca Dataset

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import re
from datetime import datetime

# Hubungkan Google Drive
drive.mount('/content/drive')
# Lokasi dataset kotor di Google Drive
file_path = '/content/drive/MyDrive/2418040_Data Mining/Data/Dataset Kotor Layanan ART.xlsx'
# Membaca dataset
df = pd.read_excel(file_path)

print("Dataset berhasil dibaca!")
print("Jumlah baris :", len(df))
print("Jumlah kolom:", len(df.columns))

display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset berhasil dibaca!
Jumlah baris : 50
Jumlah kolom: 9


,No,Nama,Alamat,No. Telepon,Tanggal,Jam/Waktu Layanan,Jam Kerja,Jenis Layanan,Status Layanan
0,1,Himawan Rakha,Jl. Mawar Blok A-12,081234567890,25/09/2026,08:30,4 jam,Bersih-bersih,Diterima
1,2,SITI aminah,"Jl. Soekarno Hatta, Blok B/45",+62 812-3456-7891,26-9-2026,1:00 PM,8 Jam,Paket Lengkap,diterima
2,3,BUDI Santoso,Jl.Kenanga Blok C No.7,0822 9876 5432,27/09/26,14.15 WIB,2,memasak,Ditolak
3,4,AGUS SETIAWAN,Jalan Kamboja Blok A-15,083 145 678 90,5 Oktober 2026,09.30,3 Jam,Bersih bersih,DITERIMA
4,5,Fajar Ramadhan,"Jalan Melati, Blok E/09",628123456789,29/9/2026,Jam 16:45 WIB,05 jam,paket lengkap,ditolak


# 2. Cek Struktur Dataset

In [ ]:
print("Nama kolom:")
for kolom in df.columns:
    print("-", kolom)

print("\nUkuran dataset:", df.shape)

Nama kolom:
- No
- Nama
- Alamat
- No. Telepon
- Tanggal
- Jam/Waktu Layanan
- Jam Kerja
- Jenis Layanan
- Status Layanan

Ukuran dataset: (50, 9)


# 3. Menyimpan Salinan Data Kotor

In [ ]:
# Menyimpan salinan dataset kotor
df_kotor = df.copy()

# 4. Cleansing Kolom Nama

In [ ]:
def clean_nama(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = re.sub(r'\s+', ' ', value)

    return value.title()


df['Nama'] = df['Nama'].apply(clean_nama)

print("Nama kotor:")
display(df_kotor[['Nama']].head(10))

print("Nama bersih:")
display(df[['Nama']].head(10))

Nama kotor:


,Nama
0,Himawan Rakha
1,SITI aminah
2,BUDI Santoso
3,AGUS SETIAWAN
4,Fajar Ramadhan
5,DEWI LESTARI
6,Andi PRATAMA
7,NURUL HIDAYAH
8,rudi HARTONO
9,MAYA SARI


Nama bersih:


,Nama
0,Himawan Rakha
1,Siti Aminah
2,Budi Santoso
3,Agus Setiawan
4,Fajar Ramadhan
5,Dewi Lestari
6,Andi Pratama
7,Nurul Hidayah
8,Rudi Hartono
9,Maya Sari


# 5. Cleansing Kolom Alamat

In [ ]:
def clean_alamat(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip()

    # Hilangkan spasi berlebih
    s = re.sub(r'\s+', ' ', s)

    # Samakan "Jalan" menjadi "Jl."
    s = re.sub(r'(?i)^jalan\b', 'Jl.', s)

    # Samakan variasi Jl / Jl. / Jl. tanpa spasi
    s = re.sub(r'(?i)^jl\.?\s*', 'Jl. ', s)

    # Rapikan koma
    s = re.sub(r'\s*,\s*', ', ', s)

    # Format Blok
    s = re.sub(
        r'(?i)\bblok\s*([A-Za-z])\s*[-/]?\s*(?:no\.?\s*)?(\d+)\b',
        lambda m: f'Blok {m.group(1).upper()} No. {int(m.group(2))}',
        s
    )

    # Jika sudah memiliki No.
    s = re.sub(
        r'(?i)\s*(?:no\.?|nomor)\s*[-:]?\s*(\d+)\b',
        lambda m: f' No. {int(m.group(1))}',
        s
    )

    # Jika tidak memiliki blok dan nomor belum ditemukan,
    # ambil angka terakhir sebagai nomor rumah
    if not re.search(r'(?i)\bBlok\b', s):
        if not re.search(r'(?i)\bNo\.\s*\d+', s):
            match = re.search(r'(\d+)\s*$', s)

            if match:
                nomor = int(match.group(1))
                jalan = s[:match.start()].rstrip(' ,')
                s = f'{jalan}, No. {nomor}'

    # Jika memiliki Blok, pastikan dipisahkan dengan koma
    if re.search(r'(?i)\bBlok\b', s):
        s = re.sub(r'\s*,?\s*(Blok)', r', \1', s, flags=re.I)

    # Rapikan koma ganda
    s = re.sub(r',\s*,', ', ', s)

    return s.strip()


df['Alamat'] = df['Alamat'].apply(clean_alamat)

print("Alamat kotor:")
display(df_kotor[['Alamat']].head(10))

print("Alamat bersih:")
display(df[['Alamat']].head(10))

Alamat kotor:


,Alamat
0,Jl. Mawar Blok A-12
1,"Jl. Soekarno Hatta, Blok B/45"
2,Jl.Kenanga Blok C No.7
3,Jalan Kamboja Blok A-15
4,"Jalan Melati, Blok E/09"
5,Jl. Anggrek Blok A No 8
6,"Jl. Diponegoro, Blok C No 3"
7,Jl. Cempaka Blok B-11
8,Jl. Merpati No. 22
9,Jalan Flamboyan Blok D No. 4


Alamat bersih:


,Alamat
0,"Jl. Mawar, Blok A No. 12"
1,"Jl. Soekarno Hatta, Blok B No. 45"
2,"Jl. Kenanga, Blok C No. 7"
3,"Jl. Kamboja, Blok A No. 15"
4,"Jl. Melati, Blok E No. 9"
5,"Jl. Anggrek, Blok A No. 8"
6,"Jl. Diponegoro, Blok C No. 3"
7,"Jl. Cempaka, Blok B No. 11"
8,Jl. Merpati No. 22
9,"Jl. Flamboyan, Blok D No. 4"


# 6. Cleansing Kolom No. Telepon

In [ ]:
def clean_telepon(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip()

    # Hapus semua karakter selain angka
    s = re.sub(r'\D', '', s)

    # Jika menggunakan kode negara 62
    if s.startswith('62'):
        s = '0' + s[2:]

    # Jika tidak diawali 0
    elif not s.startswith('0'):
        s = '0' + s

    return s


df['No. Telepon'] = df['No. Telepon'].apply(clean_telepon)

print("No. Telepon kotor:")
display(df_kotor[['No. Telepon']].head(10))

print("No. Telepon bersih:")
display(df[['No. Telepon']].head(10))

No. Telepon kotor:


,No. Telepon
0,081234567890
1,+62 812-3456-7891
2,0822 9876 5432
3,083 145 678 90
4,628123456789
5,0812 7788 9900
6,0857-1234-5678
7,+62813 4567 8899
8,0813-99887766
9,0821 3344 5566


No. Telepon bersih:


,No. Telepon
0,081234567890
1,081234567891
2,082298765432
3,08314567890
4,08123456789
5,081277889900
6,085712345678
7,081345678899
8,081399887766
9,082133445566


# 7. Cleansing Kolom Tanggal

In [ ]:
bulan = {
    'januari': '01',
    'februari': '02',
    'maret': '03',
    'april': '04',
    'mei': '05',
    'juni': '06',
    'juli': '07',
    'agustus': '08',
    'september': '09',
    'oktober': '10',
    'november': '11',
    'desember': '12'
}


def clean_tanggal(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower()

    # Ubah nama bulan menjadi angka
    for nama_bulan, nomor_bulan in bulan.items():
        s = re.sub(
            rf'\b{nama_bulan}\b',
            nomor_bulan,
            s
        )

    # Konversi tanggal
    tanggal = pd.to_datetime(
        s,
        dayfirst=True,
        errors='coerce'
    )

    if pd.isna(tanggal):
        return np.nan

    return tanggal.strftime('%d-%m-%Y')


df['Tanggal'] = df['Tanggal'].apply(clean_tanggal)

print("Tanggal kotor:")
display(df_kotor[['Tanggal']].head(10))

print("Tanggal bersih:")
display(df[['Tanggal']].head(10))

Tanggal kotor:


,Tanggal
0,25/09/2026
1,26-9-2026
2,27/09/26
3,5 Oktober 2026
4,29/9/2026
5,30/09/2026
6,01-10-2026
7,2/10/26
8,03 Oktober 2026
9,04/10/2026


Tanggal bersih:


,Tanggal
0,25-09-2026
1,26-09-2026
2,27-09-2026
3,05-10-2026
4,29-09-2026
5,30-09-2026
6,01-10-2026
7,02-10-2026
8,03-10-2026
9,04-10-2026


# 8. Cleansing Kolom Jam/Waktu Layanan

In [ ]:
def clean_waktu(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip().upper()

    # Hilangkan kata JAM
    s = re.sub(r'\bJAM\b', '', s)

    # Hilangkan keterangan Waktu
    s = re.sub(r'\bWIB\b', '', s)

    s = s.strip()

    # Ubah titik menjadi titik dua
    s = s.replace('.', ':')

    # Konversi ke waktu
    waktu = pd.to_datetime(
        s,
        errors='coerce'
    )

    if pd.isna(waktu):
        return np.nan

    return waktu.strftime('%H:%M')


df['Jam/Waktu Layanan'] = df['Jam/Waktu Layanan'].apply(clean_waktu)

print("Jam/Waktu Layanan kotor:")
display(df_kotor[['Jam/Waktu Layanan']].head(10))

print("Jam/Waktu Layanan bersih:")
display(df[['Jam/Waktu Layanan']].head(10))

Jam/Waktu Layanan kotor:


,Jam/Waktu Layanan
0,08:30
1,1:00 PM
2,14.15 WIB
3,09.30
4,Jam 16:45 WIB
5,07:30
6,10.00 WIB
7,13:15
8,08.45 WIB
9,15:00


Jam/Waktu Layanan bersih:


,Jam/Waktu Layanan
0,08:30
1,13:00
2,14:15
3,09:30
4,16:45
5,07:30
6,10:00
7,13:15
8,08:45
9,15:00


# 9. Cleansing Kolom Jam Kerja

In [ ]:
def clean_jam_kerja(value):
    if pd.isna(value):
        return np.nan

    s = str(value)

    # Ambil angka saja
    match = re.search(r'(\d+(?:[.,]\d+)?)', s)

    if match:
        return float(match.group(1))

    return np.nan


df['Jam Kerja'] = df['Jam Kerja'].apply(clean_jam_kerja)

print("Jam Kerja kotor:")
display(df_kotor[['Jam Kerja']].head(10))

print("Jam Kerja bersih:")
display(df[['Jam Kerja']].head(10))

Jam Kerja kotor:


,Jam Kerja
0,4 jam
1,8 Jam
2,2
3,3 Jam
4,05 jam
5,5 Jam
6,1
7,7 jam
8,3 Jam
9,3 jam kerja


Jam Kerja bersih:


,Jam Kerja
0,4.0
1,8.0
2,2.0
3,3.0
4,5.0
5,5.0
6,1.0
7,7.0
8,3.0
9,3.0


# 10. Cleansing Kolom Jenis Layanan

In [ ]:
def clean_jenis_layanan(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower()

    s = re.sub(r'[\s\-]+', ' ', s)

    if 'bersih' in s:
        return 'Bersih-bersih'

    elif 'masak' in s:
        return 'Memasak'

    elif 'paket' in s:
        return 'Paket lengkap'

    return np.nan


df['Jenis Layanan'] = df['Jenis Layanan'].apply(clean_jenis_layanan)

print("Jenis Layanan kotor:")
display(df_kotor[['Jenis Layanan']].head(10))

print("Jenis Layanan bersih:")
display(df[['Jenis Layanan']].head(10))

Jenis Layanan kotor:


,Jenis Layanan
0,Bersih-bersih
1,Paket Lengkap
2,memasak
3,Bersih bersih
4,paket lengkap
5,Bersih-Bersih
6,Memasak
7,paket lengkap
8,Bersih-bersih
9,MEMASAK


Jenis Layanan bersih:


,Jenis Layanan
0,Bersih-bersih
1,Paket lengkap
2,Memasak
3,Bersih-bersih
4,Paket lengkap
5,Bersih-bersih
6,Memasak
7,Paket lengkap
8,Bersih-bersih
9,Memasak


# 11. Cleansing Kolom Status Layanan

In [ ]:
def clean_status(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower()

    if 'terima' in s:
        return 'Diterima'

    elif 'tolak' in s:
        return 'Ditolak'

    return np.nan


df['Status Layanan'] = df['Status Layanan'].apply(clean_status)

print("Status Layanan kotor:")
display(df_kotor[['Status Layanan']].head(10))

print("Status Layanan bersih:")
display(df[['Status Layanan']].head(10))

Status Layanan kotor:


,Status Layanan
0,Diterima
1,diterima
2,Ditolak
3,DITERIMA
4,ditolak
5,Diterima
6,diterima
7,Ditolak
8,DITERIMA
9,diterima


Status Layanan bersih:


,Status Layanan
0,Diterima
1,Diterima
2,Ditolak
3,Diterima
4,Ditolak
5,Diterima
6,Diterima
7,Ditolak
8,Diterima
9,Diterima


# 12. Pengecekan Data Yang Hilang

In [ ]:
kolom_data = [
    'Nama',
    'Alamat',
    'No. Telepon',
    'Tanggal',
    'Jam/Waktu Layanan',
    'Jam Kerja',
    'Jenis Layanan',
    'Status Layanan'
]

data_hilang = df[df[kolom_data].isna().any(axis=1)].copy()

print("Jumlah baris yang memiliki data hilang:", len(data_hilang))

if len(data_hilang) > 0:
    display(data_hilang)
else:
    print("Tidak terdapat data yang hilang.")

Jumlah baris yang memiliki data hilang: 3


,No,Nama,Alamat,No. Telepon,Tanggal,Jam/Waktu Layanan,Jam Kerja,Jenis Layanan,Status Layanan
23,24,Fani Aprilia,"Jl. Durian, Blok E No. 12",081324681357,18-10-2026,09:00,NaN,Paket lengkap,Diterima
35,36,Vina Kusuma,"Jl. Nangka, Blok A No. 21",083899887766,30-10-2026,07:30,NaN,Paket lengkap,Ditolak
47,48,Erna Wulandari,"Jl. Cempaka, Blok E No. 15",08521234890,11-11-2026,10:00,NaN,Paket lengkap,Ditolak


# 13. Mengisi Nilai Jam Kerja

In [ ]:
minimum_jam = (
    df.groupby('Jenis Layanan')['Jam Kerja']
    .min()
)

print("Minimum Jam Kerja setiap layanan:")
display(minimum_jam)

# Simpan baris yang memiliki Jam Kerja kosong
data_jam_hilang = df[df['Jam Kerja'].isna()].copy()

# Simpan kondisi sebelum diperbaiki
data_sebelum_isi = data_jam_hilang.copy()

# Isi Jam Kerja berdasarkan minimum layanan
df.loc[df['Jam Kerja'].isna(), 'Jam Kerja'] = (
    df.loc[df['Jam Kerja'].isna(), 'Jenis Layanan']
    .map(minimum_jam)
)

# Data setelah diperbaiki
data_sesudah_isi = df.loc[
    data_sebelum_isi.index
].copy()

print("Data sebelum pengisian:")
display(data_sebelum_isi)

print("Data setelah pengisian:")
display(data_sesudah_isi)

Minimum Jam Kerja setiap layanan:


,Jam Kerja
Jenis Layanan,
Bersih-bersih,3.0
Memasak,1.0
Paket lengkap,5.0


Data sebelum pengisian:


,No,Nama,Alamat,No. Telepon,Tanggal,Jam/Waktu Layanan,Jam Kerja,Jenis Layanan,Status Layanan
23,24,Fani Aprilia,"Jl. Durian, Blok E No. 12",081324681357,18-10-2026,09:00,NaN,Paket lengkap,Diterima
35,36,Vina Kusuma,"Jl. Nangka, Blok A No. 21",083899887766,30-10-2026,07:30,NaN,Paket lengkap,Ditolak
47,48,Erna Wulandari,"Jl. Cempaka, Blok E No. 15",08521234890,11-11-2026,10:00,NaN,Paket lengkap,Ditolak


Data setelah pengisian:


,No,Nama,Alamat,No. Telepon,Tanggal,Jam/Waktu Layanan,Jam Kerja,Jenis Layanan,Status Layanan
23,24,Fani Aprilia,"Jl. Durian, Blok E No. 12",081324681357,18-10-2026,09:00,5.0,Paket lengkap,Diterima
35,36,Vina Kusuma,"Jl. Nangka, Blok A No. 21",083899887766,30-10-2026,07:30,5.0,Paket lengkap,Ditolak
47,48,Erna Wulandari,"Jl. Cempaka, Blok E No. 15",08521234890,11-11-2026,10:00,5.0,Paket lengkap,Ditolak


# 14. Pengecekan Data Duplikat

In [ ]:
kolom_duplikat = [
    'Nama',
    'Alamat',
    'No. Telepon',
    'Tanggal',
    'Jam/Waktu Layanan',
    'Jam Kerja',
    'Jenis Layanan'
]

# Cari baris yang memiliki nilai sama
duplikat = df[
    df.duplicated(
        subset=kolom_duplikat,
        keep=False
    )
].copy()

print("Data yang terdeteksi duplikat:")
if len(duplikat) > 0:
    display(duplikat)
else:
    print("Tidak terdapat data duplikat.")

Data yang terdeteksi duplikat:


,No,Nama,Alamat,No. Telepon,Tanggal,Jam/Waktu Layanan,Jam Kerja,Jenis Layanan,Status Layanan
3,4,Agus Setiawan,"Jl. Kamboja, Blok A No. 15",08314567890,05-10-2026,09:30,3.0,Bersih-bersih,Diterima
10,11,Agus Setiawan,"Jl. Kamboja, Blok A No. 15",08314567890,05-10-2026,09:30,3.0,Bersih-bersih,Ditolak


# 15. Perbandingan Data Duplikat

In [ ]:
if len(duplikat) > 0:

    nomor_duplikat = duplikat['No'].tolist()

    data_kotor_duplikat = df_kotor[
        df_kotor['No'].isin(nomor_duplikat)
    ].copy()

    data_bersih_duplikat = df[
        df['No'].isin(nomor_duplikat)
    ].copy()

    print("BARIS KOTOR YANG DUPLIKAT")
    display(data_kotor_duplikat)

    print("BARIS BERSIH")
    display(data_bersih_duplikat)

else:
    print("Tidak ada baris duplikat yang dapat dibandingkan.")

BARIS KOTOR YANG DUPLIKAT


,No,Nama,Alamat,No. Telepon,Tanggal,Jam/Waktu Layanan,Jam Kerja,Jenis Layanan,Status Layanan
3,4,AGUS SETIAWAN,Jalan Kamboja Blok A-15,083 145 678 90,5 Oktober 2026,09.30,3 Jam,Bersih bersih,DITERIMA
10,11,Agus Setiawan,Jl. Kamboja Blok A No 15,0831 4567 890,5-10-2026,09.30 WIB,3,Bersih bersih,Ditolak


BARIS BERSIH


,No,Nama,Alamat,No. Telepon,Tanggal,Jam/Waktu Layanan,Jam Kerja,Jenis Layanan,Status Layanan
3,4,Agus Setiawan,"Jl. Kamboja, Blok A No. 15",08314567890,05-10-2026,09:30,3.0,Bersih-bersih,Diterima
10,11,Agus Setiawan,"Jl. Kamboja, Blok A No. 15",08314567890,05-10-2026,09:30,3.0,Bersih-bersih,Ditolak


# 16. Menghapus Data Duplikat

In [ ]:
jumlah_sebelum = len(df)

df_bersih = df.drop_duplicates(
    subset=kolom_duplikat,
    keep='first'
).copy()

jumlah_sesudah = len(df_bersih)

print("Jumlah data sebelum menghapus duplikat :", jumlah_sebelum)
print("Jumlah data setelah menghapus duplikat :", jumlah_sesudah)
print("Jumlah duplikat yang dihapus           :", jumlah_sebelum - jumlah_sesudah)

Jumlah data sebelum menghapus duplikat : 50
Jumlah data setelah menghapus duplikat : 49
Jumlah duplikat yang dihapus           : 1


# 17. Mengurutkan Nomer Data

In [ ]:
df_bersih = df_bersih.reset_index(drop=True)

# Membuat nomor urut baru mulai dari 1
df_bersih['No'] = range(1, len(df_bersih) + 1)

print("Nomor data berhasil diperbarui.")
display(df_bersih.head(10))

Nomor data berhasil diperbarui.


,No,Nama,Alamat,No. Telepon,Tanggal,Jam/Waktu Layanan,Jam Kerja,Jenis Layanan,Status Layanan
0,1,Himawan Rakha,"Jl. Mawar, Blok A No. 12",081234567890,25-09-2026,08:30,4.0,Bersih-bersih,Diterima
1,2,Siti Aminah,"Jl. Soekarno Hatta, Blok B No. 45",081234567891,26-09-2026,13:00,8.0,Paket lengkap,Diterima
2,3,Budi Santoso,"Jl. Kenanga, Blok C No. 7",082298765432,27-09-2026,14:15,2.0,Memasak,Ditolak
3,4,Agus Setiawan,"Jl. Kamboja, Blok A No. 15",08314567890,05-10-2026,09:30,3.0,Bersih-bersih,Diterima
4,5,Fajar Ramadhan,"Jl. Melati, Blok E No. 9",08123456789,29-09-2026,16:45,5.0,Paket lengkap,Ditolak
5,6,Dewi Lestari,"Jl. Anggrek, Blok A No. 8",081277889900,30-09-2026,07:30,5.0,Bersih-bersih,Diterima
6,7,Andi Pratama,"Jl. Diponegoro, Blok C No. 3",085712345678,01-10-2026,10:00,1.0,Memasak,Diterima
7,8,Nurul Hidayah,"Jl. Cempaka, Blok B No. 11",081345678899,02-10-2026,13:15,7.0,Paket lengkap,Ditolak
8,9,Rudi Hartono,Jl. Merpati No. 22,081399887766,03-10-2026,08:45,3.0,Bersih-bersih,Diterima
9,10,Maya Sari,"Jl. Flamboyan, Blok D No. 4",082133445566,04-10-2026,15:00,3.0,Memasak,Diterima


# 18. Validasi Data

In [ ]:
print("========== HASIL VALIDASI ==========")

# 1. Validasi Nama
nama_valid = (
    df_bersih['Nama'].notna() &
    ~df_bersih['Nama'].astype(str).str.contains(r'\s{2,}', regex=True)
)

print("Nama valid              :", nama_valid.all())

# 2. Validasi Alamat
alamat_valid = (
    df_bersih['Alamat'].notna() &
    df_bersih['Alamat'].astype(str).str.startswith('Jl.')
)

print("Alamat valid            :", alamat_valid.all())

# 3. Validasi No. Telepon
telepon_valid = (
    df_bersih['No. Telepon'].notna() &
    df_bersih['No. Telepon'].astype(str).str.match(r'^0\d+$')
)

print("No. Telepon valid       :", telepon_valid.all())

# 4. Validasi Tanggal
tanggal_valid = (
    df_bersih['Tanggal'].notna() &
    df_bersih['Tanggal'].astype(str).str.match(r'^\d{2}-\d{2}-\d{4}$')
)

print("Tanggal valid           :", tanggal_valid.all())

# 5. Validasi Waktu
waktu_valid = (
    df_bersih['Jam/Waktu Layanan'].notna() &
    df_bersih['Jam/Waktu Layanan'].astype(str).str.match(r'^\d{2}:\d{2}$')
)

print("Jam/Waktu valid         :", waktu_valid.all())

# 6. Validasi Jam Kerja
jam_valid = (
    df_bersih['Jam Kerja'].notna() &
    (df_bersih['Jam Kerja'] > 0)
)

print("Jam Kerja valid         :", jam_valid.all())

# 7. Validasi Jenis Layanan
layanan_valid = df_bersih['Jenis Layanan'].isin([
    'Bersih-bersih',
    'Memasak',
    'Paket lengkap'
])

print("Jenis Layanan valid     :", layanan_valid.all())

# 8. Validasi Status
status_valid = df_bersih['Status Layanan'].isin([
    'Diterima',
    'Ditolak'
])

print("Status Layanan valid    :", status_valid.all())

# 9. Validasi data kosong
tidak_ada_kosong = not df_bersih[kolom_data].isna().any().any()

print("Tidak ada data kosong   :", tidak_ada_kosong)

========== HASIL VALIDASI ==========
Nama valid              : True
Alamat valid            : True
No. Telepon valid       : True
Tanggal valid           : True
Jam/Waktu valid         : True
Jam Kerja valid         : True
Jenis Layanan valid     : True
Status Layanan valid    : True
Tidak ada data kosong   : True


# 19. Menampilkan Sebagian Hasil Akhir

In [ ]:
print("========== 10 DATA PERTAMA HASIL CLEANSING ==========")

display(df_bersih.head(10))

========== 10 DATA PERTAMA HASIL CLEANSING ==========


,No,Nama,Alamat,No. Telepon,Tanggal,Jam/Waktu Layanan,Jam Kerja,Jenis Layanan,Status Layanan
0,1,Himawan Rakha,"Jl. Mawar, Blok A No. 12",081234567890,25-09-2026,08:30,4.0,Bersih-bersih,Diterima
1,2,Siti Aminah,"Jl. Soekarno Hatta, Blok B No. 45",081234567891,26-09-2026,13:00,8.0,Paket lengkap,Diterima
2,3,Budi Santoso,"Jl. Kenanga, Blok C No. 7",082298765432,27-09-2026,14:15,2.0,Memasak,Ditolak
3,4,Agus Setiawan,"Jl. Kamboja, Blok A No. 15",08314567890,05-10-2026,09:30,3.0,Bersih-bersih,Diterima
4,5,Fajar Ramadhan,"Jl. Melati, Blok E No. 9",08123456789,29-09-2026,16:45,5.0,Paket lengkap,Ditolak
5,6,Dewi Lestari,"Jl. Anggrek, Blok A No. 8",081277889900,30-09-2026,07:30,5.0,Bersih-bersih,Diterima
6,7,Andi Pratama,"Jl. Diponegoro, Blok C No. 3",085712345678,01-10-2026,10:00,1.0,Memasak,Diterima
7,8,Nurul Hidayah,"Jl. Cempaka, Blok B No. 11",081345678899,02-10-2026,13:15,7.0,Paket lengkap,Ditolak
8,9,Rudi Hartono,Jl. Merpati No. 22,081399887766,03-10-2026,08:45,3.0,Bersih-bersih,Diterima
9,10,Maya Sari,"Jl. Flamboyan, Blok D No. 4",082133445566,04-10-2026,15:00,3.0,Memasak,Diterima


# 20. Menyimpan Hasil Akhir

In [ ]:
output_path = '/content/drive/MyDrive/2418040_Data Mining/Data/Dataset Bersih Layanan ART.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

  df_bersih.to_excel(
        writer,
        sheet_name='Data Bersih',
        index=False
    )

print("========================================")
print("DATA BERHASIL DISIMPAN")
print("========================================")

DATA BERHASIL DISIMPAN
